# 🌾 Seasonal Agriculture Performance Analysis
### Major Project • VOIS AICTE Batch 1 (2026–2027)

**Project focus:** Evidence-based comparison of agricultural performance across **Kharif, Rabi and Zaid** seasons.

---

### What this notebook delivers
- Dataset understanding and quality checks
- Reproducible data cleaning
- Seasonal performance comparison
- Crop and regional analysis
- Environmental and resource analysis
- Correlation analysis
- ANOVA, Kruskal–Wallis and Chi-square testing
- Clear findings, recommendations and limitations

> **Interpretation rule:** statistical association is not treated as proof of causation.


## 1. Problem Statement

Agricultural activities are influenced by seasonal environmental conditions, farming practices, resource availability and market conditions. The purpose of this project is to investigate how agricultural performance changes across seasons and to identify meaningful patterns, trends, relationships and variations in the available data.


## 2. Objectives

1. Explore and understand the dataset.
2. Clean and prepare the data for analysis.
3. Compare agricultural performance across seasons.
4. Investigate environmental conditions and resource usage.
5. Examine economic outcomes.
6. Identify important relationships and unusual patterns.
7. Apply suitable statistical techniques.
8. Produce evidence-based conclusions and recommendations.


## 3. Analytical Questions

**Q1.** How does yield vary across seasons?  
**Q2.** How does profitability vary across seasons?  
**Q3.** Which season performs best economically?  
**Q4.** How does water efficiency differ by season?  
**Q5.** Are seasonal differences statistically significant?  
**Q6.** Does irrigation-method distribution differ by season?  
**Q7.** Which crops and states have stronger average profitability?  
**Q8.** Which numerical variables are most associated with profit?


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f_oneway, kruskal, chi2_contingency

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
print("Dataset shape:", df.shape)
display(df.head())


## 4. Dataset Understanding

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Unique Values": df.nunique(),
    "Missing Values": df.isna().sum()
}))

print("Duplicate rows:", df.duplicated().sum())


### Data-cleaning strategy

Missing numeric observations are handled using the **median of the corresponding season**. This keeps the imputation aligned with the seasonal structure of the project rather than imposing one global value across all seasons. Duplicate rows are checked separately.


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df.groupby("Season")[col].transform(
            lambda s: s.fillna(s.median())
        )

print("Remaining missing values:", int(df.isna().sum().sum()))


## 5. Descriptive Statistics

In [ ]:
display(df.describe(include="all").T)


## 6. Seasonal Performance Dashboard

In [ ]:
season_summary = df.groupby("Season").agg(
    Records=("Farm_ID","count"),
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Production=("Production_Tonnes","mean"),
    Avg_Revenue=("Revenue_INR","mean"),
    Avg_Cost=("Total_Cost_INR","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Disease_Risk=("Disease_Pest_Risk_pct","mean"),
    Avg_Rainfall=("Rainfall_mm","mean")
).round(2)

display(season_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
season_summary["Avg_Yield"].plot(kind="bar", ax=ax)
ax.set_title("Average Yield by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Yield (tonnes/ha)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
season_summary["Avg_Profit"].plot(kind="bar", ax=ax)
ax.set_title("Average Profit by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Profit (INR)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
season_summary[["Avg_Revenue","Avg_Cost"]].plot(kind="bar", ax=ax)
ax.set_title("Average Revenue vs Cost by Season")
ax.set_xlabel("Season")
ax.set_ylabel("INR")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
season_summary["Avg_Water_Efficiency"].plot(kind="bar", ax=ax)
ax.set_title("Water Efficiency by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Tonnes per 1,000 m³")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 7. Environmental and Resource Analysis

In [ ]:
environment_summary = df.groupby("Season").agg(
    Rainfall_mm=("Rainfall_mm","mean"),
    Temperature_C=("Avg_Temperature_C","mean"),
    Humidity_pct=("Humidity_pct","mean"),
    Soil_Moisture_pct=("Soil_Moisture_pct","mean"),
    Fertilizer_kg_ha=("Fertilizer_kg_ha","mean"),
    Pesticide_Litre_ha=("Pesticide_Litre_ha","mean"),
    Water_Used_m3=("Water_Used_m3","mean"),
    Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean")
).round(2)

display(environment_summary)


## 8. Crop-wise Analysis

In [ ]:
crop_summary = df.groupby("Crop").agg(
    Records=("Farm_ID","count"),
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean")
).sort_values("Avg_Profit", ascending=False).round(2)

display(crop_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
crop_summary["Avg_Profit"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Average Profit by Crop")
ax.set_xlabel("Average Profit (INR)")
ax.set_ylabel("Crop")
plt.tight_layout()
plt.show()


## 9. State-wise Analysis

In [ ]:
state_summary = df.groupby("State").agg(
    Records=("Farm_ID","count"),
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean")
).sort_values("Avg_Profit", ascending=False).round(2)

display(state_summary)


## 10. Irrigation Analysis

In [ ]:
irrigation_counts = pd.crosstab(df["Season"], df["Irrigation_Method"])
irrigation_percent = pd.crosstab(
    df["Season"], df["Irrigation_Method"], normalize="index"
) * 100

display(irrigation_counts)
display(irrigation_percent.round(2))


In [ ]:
chi2, p, dof, expected = chi2_contingency(irrigation_counts)
print(f"Chi-square statistic: {chi2:.3f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

if p < 0.05:
    print("Conclusion: irrigation-method distribution differs significantly by season.")
else:
    print("Conclusion: no statistically significant seasonal difference was detected.")


## 11. Correlation Analysis

In [ ]:
corr = df.select_dtypes(include=np.number).corr()

fig, ax = plt.subplots(figsize=(12,9))
im = ax.imshow(corr, aspect="auto")
fig.colorbar(im, ax=ax, label="Correlation")
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

display(corr["Profit_INR"].sort_values(ascending=False).to_frame("Correlation_with_Profit"))


## 12. Statistical Testing

### Why three tests?

- **One-way ANOVA:** compares mean values across the three seasons.
- **Kruskal–Wallis:** non-parametric alternative that compares distributions/ranks.
- **Chi-square:** tests whether categorical irrigation-method distribution is associated with season.

The significance level is **α = 0.05**.


In [ ]:
results = []

for variable in [
    "Yield_Tonnes_Ha",
    "Profit_INR",
    "Revenue_INR",
    "Total_Cost_INR",
    "Water_Efficiency_t_per_1000m3"
]:
    groups = [g[variable].dropna().values for _, g in df.groupby("Season")]
    anova = f_oneway(*groups)
    kw = kruskal(*groups)
    results.append({
        "Variable": variable,
        "ANOVA_F": anova.statistic,
        "ANOVA_p": anova.pvalue,
        "Kruskal_H": kw.statistic,
        "Kruskal_p": kw.pvalue
    })

test_results = pd.DataFrame(results)
display(test_results.round(6))


## 13. Evidence-based Findings

In [ ]:
best_yield = season_summary["Avg_Yield"].idxmax()
best_profit = season_summary["Avg_Profit"].idxmax()
best_water = season_summary["Avg_Water_Efficiency"].idxmax()
lowest_profit = season_summary["Avg_Profit"].idxmin()

print(f"• Highest average yield: {best_yield}")
print(f"• Highest average profit: {best_profit}")
print(f"• Highest water efficiency: {best_water}")
print(f"• Lowest average profit: {lowest_profit}")
print(f"• Irrigation chi-square p-value: {p:.6f}")


## 14. Key Interpretation

- **Kharif** records the strongest average yield and average profit in the supplied dataset.
- **Zaid** records the lowest average profit and lowest average water efficiency.
- Profit is strongly linked with revenue; yield and water efficiency also show meaningful positive relationships with profit.
- The irrigation-method chi-square test does **not** provide evidence of a statistically significant seasonal difference at α = 0.05.
- Crop and state averages differ, supporting localized agricultural planning.
- Statistical significance should be interpreted as evidence of differences in this dataset, not as proof of causal mechanisms.


## 15. Recommendations

1. Use **season-specific crop and input planning**.
2. Investigate cost and market-price drivers in low-profit seasons.
3. Improve water-management efficiency where output per unit of water is weak.
4. Evaluate crops using **profit, yield, cost, price and water requirements together**.
5. Monitor environmental and disease/pest conditions as part of seasonal planning.
6. Adapt recommendations to local/state conditions.
7. Collect multi-year data to strengthen seasonal trend analysis.


## 16. Limitations

- The analysis is observational and does not establish causation.
- Missing numeric values were imputed using seasonal medians.
- Results describe the supplied dataset and should not automatically be generalized to all agricultural settings.
- A longer multi-year dataset would provide stronger evidence about recurring seasonal trends.


## 17. Conclusion

The supplied data shows meaningful seasonal variation in agricultural economic performance and resource efficiency. **Kharif emerges as the strongest overall season by average yield and profitability, while Zaid requires closer attention because of lower average profitability and water efficiency.** Statistical analysis strengthens the evidence for seasonal differences in several economic/resource measures.

Overall, the project demonstrates how data analytics can convert raw agricultural observations into actionable, evidence-based planning insights.


## 18. Project Submission Checklist

✅ Problem statement and objectives  
✅ Dataset understanding  
✅ Data cleaning and missing-value treatment  
✅ Descriptive statistics  
✅ Seasonal comparison  
✅ Environmental/resource analysis  
✅ Crop and state comparison  
✅ Irrigation analysis  
✅ Correlation analysis  
✅ Statistical tests  
✅ Visualizations  
✅ Findings and interpretation  
✅ Recommendations  
✅ Limitations  
✅ Conclusion  
